# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [38]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [39]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue is ${total_revenue:.2f} and the total units sold are {total_units}.")

The total revenue is $8520.00 and the total units sold are 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [40]:
# TODO
by_category = df.groupby('category').agg(
    revenue=('revenue', 'sum'),
    units=('qty', 'sum')
)

total_revenue = by_category['revenue'].sum()
by_category['share'] = by_category['revenue'] / total_revenue * 100

print(f"Food leads with ${by_category.loc['Food','revenue']:.2f} in revenue ({by_category.loc['Food','share']:.1f}% of the total), while RainGear brings up the rear at ${by_category.loc['RainGear','revenue']:.2f} ({by_category.loc['RainGear','share']:.1f}%).")
by_category.sort_values('revenue', ascending=False)



Food leads with $4293.00 in revenue (50.4% of the total), while RainGear brings up the rear at $901.50 (10.6%).


,revenue,units,share
category,,,
Food,4293.0,362,50.387324
Merch,1771.5,158,20.792254
Drink,1554.0,178,18.239437
RainGear,901.5,85,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [41]:
# TODO
by_vendor = df.groupby('vendor_id').agg(
    revenue=('revenue', 'sum'),
    orders=('qty', 'count')
)

by_vendor['avg_revenue'] = by_vendor['revenue'] / by_vendor['orders']

print(f"V-01 has the highest average order revenue at ${by_vendor.loc['V-01','avg_revenue']:.2f} over {by_vendor.loc['V-01','orders']} orders, only slightly ahead of V-18's ${by_vendor.loc['V-18','avg_revenue']:.2f} over {by_vendor.loc['V-18','orders']} orders.")
by_vendor.sort_values('avg_revenue', ascending=False)


V-01 has the highest average order revenue at $22.60 over 94 orders, only slightly ahead of V-18's $21.75 over 108 orders.


,revenue,orders,avg_revenue
vendor_id,,,
V-01,2124.0,94,22.595745
V-18,2349.0,108,21.750000
V-05,1914.0,93,20.580645
V-10,2133.0,105,20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [42]:
# TODO
merch_revenue = by_category.loc['Merch', 'revenue']
total_revenue = by_category['revenue'].sum()
merch_share = merch_revenue / total_revenue * 100

print(f"The share of revenue from Merch is {merch_share:.1f}%.")

The share of revenue from Merch is 20.8%.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [43]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
joined


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,NaN
2,V-18,Drink,3,4.5,13.5,NaN
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,NaN
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,NaN
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,NaN


**The unmatched vendor, and what I did about it:** _The unmatched vendor is V-18, which has no entry in the vendor_names lookup. I left it as NaN after the left join rather than dropping those rows, since dropping them would remove the 108 real orders from every report. In Q6, I relabeled the vendor name to "V-18"_

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [44]:
joined['vendor_name'] = joined['vendor_name'].fillna('V-18')
# TODO
pd.pivot_table(joined, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True)

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
V-18,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [45]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell the vendors to focus on selling food, since the total revenue for this category was the highest out of the four product categories for each vendor. The food category share across all vendors was over 50% (50.387), which is the majority. The fewest units sold out of all the product categories is rain gear, with a total of 85 units, which only makes about 11% of the total share of products sold.  

b) The least trustworthy answer is still question 6. After fixing the pivot so V-18's rows aren't silently dropped, the totals are now numerically complete — but V-18 still has no real name, since it's missing from the vendor_names lookup entirely. So a report that breaks revenue down 'by vendor' is showing accurate numbers next to a placeholder label for over a quarter of total revenue